# Vanguard_part4_duckdb_tool_agent

**Group:** Vanguard  
**Jaydon Hylton (2210144)** **Chadwick Cox (1800729)** **Semoy Smith (1505625)**

DuckDB CTE + `RANK() OVER (PARTITION BY major)`, `COMP 2605 >= C`, advising hold/warning, `bind_tools`, synthesized letter.

In [ ]:
import json
import re
from typing import Any

import duckdb
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

from vanguard.azure_chat import build_chat_model, project_root

# Transcript rows live in this CSV; DuckDB will query it as a table.
csv_path = project_root() / "data" / "student_transcripts.csv"
if not csv_path.is_file():
    raise FileNotFoundError(csv_path)

# Reject writes and multi-statement SQL so the model can only SELECT.
FORBIDDEN_SQL = re.compile(
    r"\b(insert|update|delete|drop|alter|create|attach|copy)\b", re.I
)
GRADE_POINTS = {
    "A": 4.0, "A-": 3.7, "B+": 3.3, "B": 3.0, "B-": 2.7,
    "C+": 2.3, "C": 2.0, "D": 1.0, "F": 0.0,
}
# Load the CSV into an in-memory DuckDB table once, then run SQL against it.
conn = duckdb.connect(":memory:")
conn.execute(
    f"CREATE TABLE student_transcripts AS SELECT * FROM read_csv_auto('{csv_path.as_posix()}', header=TRUE)"
)


def run_readonly_sql(sql: str) -> list[dict[str, Any]]:
    query = sql.strip().rstrip(";")
    if ";" in query or FORBIDDEN_SQL.search(query):
        raise ValueError("Only a single read-only SELECT is allowed")
    cur = conn.execute(query)
    cols = [d[0] for d in cur.description]
    return [dict(zip(cols, row, strict=True)) for row in cur.fetchall()]


# Argument shapes: a SQL string, a prereq check, or a registrar action.
class TranscriptSqlInput(BaseModel):
    sql: str = Field(description="Read-only SELECT on student_transcripts. Columns: student_id, student_name, major, course_code, letter_grade, quality_points, credit_hours, term, department.")


class PrerequisiteInput(BaseModel):
    student_id: str = Field(description="Nine-digit student ID")
    target_course: str = Field(description="Advanced course, e.g. COMP 3901")


class AdvisingActionInput(BaseModel):
    student_id: str = Field(description="Nine-digit student ID")
    action: str = Field(description="lift_advising_hold or issue_academic_warning")
    reason: str = Field(description="Justification")


# Run a SELECT against the transcript table and return the rows as JSON.
def execute_transcript_sql(sql: str) -> str:
    try:
        data = run_readonly_sql(sql)
        return json.dumps(
            {"tool_name": "execute_transcript_sql", "query_status": "success", "data": data},
            indent=2,
            default=str,
        )
    except Exception as exc:
        return json.dumps(
            {
                "tool_name": "execute_transcript_sql",
                "query_status": "error",
                "error": str(exc),
                "hint": (
                    "Columns are quality_points and credit_hours, not grade_point or credits. "
                    "Use ROUND(SUM(quality_points * credit_hours) / SUM(credit_hours), 2) "
                    "AS cumulative_gpa and RANK() OVER (PARTITION BY major ORDER BY cumulative_gpa DESC)."
                ),
            },
            indent=2,
        )


# Look up COMP 2605 on the transcript. A C (2.0) is the floor for COMP 3901 / COMP 3601.
def verify_prerequisite_and_standing(student_id: str, target_course: str) -> str:
    rows = run_readonly_sql(
        "SELECT course_code, letter_grade FROM student_transcripts "
        f"WHERE student_id = '{student_id.strip()}'"
    )
    grades = {r["course_code"]: GRADE_POINTS.get(r["letter_grade"], 0.0) for r in rows}
    points = grades.get("COMP 2605", 0.0)
    course = target_course.strip().upper()
    return json.dumps(
        {
            "tool_name": "verify_prerequisite_and_standing",
            "student_id": student_id,
            "target_course": course,
            "required_prerequisite": "COMP 2605",
            "minimum_letter_grade": "C",
            "comp_2605_grade_points": points,
            "eligible": course in {"COMP 3901", "COMP 3601"} and points >= 2.0,
        },
        indent=2,
    )


# Queue a registrar action: lift a registration hold, or issue a warning letter.
def advising_action_hold_override(student_id: str, action: str, reason: str) -> str:
    if action not in {"lift_advising_hold", "issue_academic_warning"}:
        raise ValueError("unsupported action")
    return json.dumps(
        {
            "tool_name": "advising_action_hold_override",
            "student_id": student_id,
            "action": action,
            "status": "queued_for_registrar",
            "reason": reason,
        },
        indent=2,
    )


# Register the functions so the chat model can call them by name.
tools = [
    StructuredTool.from_function(
        execute_transcript_sql,
        name="execute_transcript_sql",
        description="DuckDB SQL over student_transcripts using quality_points and credit_hours (not grade_point/credits). Prefer a CTE and RANK over major.",
        args_schema=TranscriptSqlInput,
    ),
    StructuredTool.from_function(
        verify_prerequisite_and_standing,
        name="verify_prerequisite_and_standing",
        description="Check COMP 2605 at C or higher before COMP 3901 / COMP 3601.",
        args_schema=PrerequisiteInput,
    ),
    StructuredTool.from_function(
        advising_action_hold_override,
        name="advising_action_hold_override",
        description="Lift an advising hold or issue an academic warning letter.",
        args_schema=AdvisingActionInput,
    ),
]
print(csv_path)
print([t.name for t in tools])


In [ ]:
# Call each tool ourselves first: GPA/rank for Mercer, then eligibility, then a warning for Lee.
print(
    execute_transcript_sql(
        """
        WITH per_student AS (
            SELECT student_id, student_name, major,
                   ROUND(SUM(quality_points * credit_hours) / SUM(credit_hours), 2) AS cumulative_gpa,
                   SUM(credit_hours) AS total_credits_earned
            FROM student_transcripts
            GROUP BY student_id, student_name, major
        )
        SELECT *,
               RANK() OVER (PARTITION BY major ORDER BY cumulative_gpa DESC) AS academic_rank_in_major
        FROM per_student
        WHERE student_id = '816001234'
        """
    )
)
print(verify_prerequisite_and_standing("816001234", "COMP 3901"))  # Mercer earned B+ in COMP 2605, so this should pass
print(verify_prerequisite_and_standing("816002345", "COMP 3901"))  # Lee earned D, so this should fail
print(
    advising_action_hold_override(
        "816002345",
        "issue_academic_warning",
        "COMP 2605 below C; not eligible for COMP 3901.",
    )
)

In [ ]:
# Attach the tools so the model can request SQL, a prereq check, or a registrar action as needed.
llm = build_chat_model("local").bind_tools(tools)
tool_map = {t.name: t for t in tools}
messages: list = [
    SystemMessage(
        content=(
            "You are a Senior Computer Science Academic Advisor. "
            "Use tools for GPA, rank, and COMP 3901 eligibility, then write a short advising letter. "
            "GPA/rank SQL must use a CTE and RANK() OVER (PARTITION BY major). "
            "SQL columns: quality_points and credit_hours (never grade_point or credits)."
        )
    ),
    HumanMessage(
        content=(
            "Review student 816001234 (Alex Mercer). "
            "Report GPA, credits, departmental rank, and COMP 3901 eligibility."
        )
    ),
]
trace: list[dict] = []
# Keep calling until the model stops asking for tools. Each result is fed back so it can write from real data.
for _ in range(6):
    ai: AIMessage = llm.invoke(messages)
    if not ai.tool_calls:
        print(json.dumps(trace, indent=2))  # structured payloads returned by the tools
        print("\n" + str(ai.content))  # advising letter written after those results
        break
    messages.append(ai)
    for call in ai.tool_calls:
        try:
            raw = tool_map[call["name"]].invoke(call["args"])
        except Exception as exc:
            raw = json.dumps({"query_status": "error", "error": str(exc)})
        trace.append({"tool_name": call["name"], "args": call["args"], "raw": json.loads(raw)})
        messages.append(ToolMessage(content=raw, tool_call_id=call["id"]))
